# Diagnóstico bug NDVI año (tiles transparentes)

Replica la construcción de `make_dashboard_html.py` y reporta dónde se está perdiendo la data.

Si este notebook abrió correctamente con tu kernel habitual, GEE ya está autenticado.

In [1]:
from pathlib import Path
import urllib.request, urllib.error
import ee, geopandas as gpd

PROJECT = 'basic-buttress-338101'
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

ROOT = Path.cwd().resolve()
while ROOT.name != 'proyecto-cgsm' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
print('ROOT:', ROOT)

AOI_PATH = ROOT / 'data' / 'raw' / 'cgsm_aoi_acotado_4326.geojson'
gdf_aoi = gpd.read_file(AOI_PATH)
if gdf_aoi.crs is None or gdf_aoi.crs.to_epsg() != 4326:
    gdf_aoi = gdf_aoi.to_crs(4326)
geom_union = gdf_aoi.geometry.union_all()
aoi = ee.Geometry(geom_union.__geo_interface__)
area_km2 = aoi.area().divide(1e6).getInfo()
print(f'AOI: {len(gdf_aoi)} polígono(s), área = {area_km2:.1f} km²')
print(f'bbox: {geom_union.bounds}')

Enter verification code:  4/1AeoWuM-Getz-BzMW_9blyJANG6mWLfMOGuO6LzESmk8G7WE0L_J__A_am8g



Successfully saved authorization token.
ROOT: /home/rstudio/work/proyecto-cgsm
AOI: 1 polígono(s), área = 839.4 km²
bbox: (-74.8496788, 10.603721099999973, -74.31747827700002, 11.125915527000018)


In [5]:
def mask_s2(image):
    qa = image.select('QA60')
    return image.updateMask(qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0)))

def add_idx(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    return image.addBands([ndvi, ndwi, ndvi.subtract(ndwi).rename('CMRI')])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2).map(add_idx))

s2_raw = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(aoi))

vis_ndvi = {'min': -0.2, 'max': 0.8,
            'palette': ['#8B0000','#D32F2F','#FF6F00','#FDD835','#7CB342','#2E7D32','#1B5E20']}

## TEST 1 — conteo de imágenes por año

In [3]:
print(f'{"Año":<6}{"Sin filtro":>14}{"CLOUDY<20":>14}{"Veredicto":>20}')
for y in range(2018, 2026):
    n_raw = s2_raw.filterDate(f'{y}-01-01', f'{y+1}-01-01').size().getInfo()
    n_filtered = s2.filterDate(f'{y}-01-01', f'{y+1}-01-01').size().getInfo()
    verdict = 'OK' if n_filtered >= 3 else ('VACÍO ❌' if n_filtered == 0 else 'POCAS ⚠')
    print(f'{y:<6}{n_raw:>14}{n_filtered:>14}{verdict:>20}')

Año       Sin filtro     CLOUDY<20           Veredicto
2018              38            36                  OK
2019             293           133                  OK
2020             281           113                  OK
2021             290           116                  OK
2022             289            83                  OK
2023             294           110                  OK
2024             294            98                  OK
2025             383           107                  OK


## TEST 2 — reducer sobre la composite mediana

In [4]:
print(f'{"Año":<6}{"NDVI_min":>12}{"NDVI_mean":>12}{"NDVI_max":>12}{"Veredicto":>20}')
for y in range(2018, 2026):
    img = (s2.filterDate(f'{y}-01-01', f'{y+1}-01-01')
           .select('NDVI').median().clip(aoi))
    try:
        stats = img.reduceRegion(
            reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), '', True),
            geometry=aoi, scale=100, maxPixels=1e8, bestEffort=True
        ).getInfo()
        nmin = stats.get('NDVI_min'); nmean = stats.get('NDVI_mean'); nmax = stats.get('NDVI_max')
        if nmean is None:
            print(f'{y:<6}{"None":>12}{"None":>12}{"None":>12}{"VACÍO ❌":>20}')
        else:
            verdict = 'OK' if -0.2 < nmean < 0.8 else 'fuera de rango'
            print(f'{y:<6}{nmin:>12.3f}{nmean:>12.3f}{nmax:>12.3f}{verdict:>20}')
    except Exception as e:
        print(f'{y:<6}   ERROR: {type(e).__name__}: {str(e)[:50]}')

Año       NDVI_min   NDVI_mean    NDVI_max           Veredicto
2018        -1.000       0.195       0.984                  OK
2019        -0.637       0.207       0.956                  OK
2020        -0.720       0.205       0.957                  OK
2021        -0.971       0.220       0.974                  OK
2022        -1.000       0.234       0.974                  OK
2023        -1.000       0.247       0.972                  OK
2024        -1.000       0.257       0.964                  OK
2025        -1.000       0.281       0.960                  OK


## TEST 3 — fetch real de un tile sobre CGSM

In [6]:
print(f'{"Año":<6}{"Tile bytes":>14}{"Veredicto":>30}')
for y in [2020, 2022, 2024]:
    img = (s2.filterDate(f'{y}-01-01', f'{y+1}-01-01')
           .select('NDVI').median().clip(aoi))
    try:
        mid = img.getMapId(vis_ndvi)
        url = (mid['tile_fetcher'].url_format
               .replace('{z}','11').replace('{x}','610').replace('{y}','927'))
        with urllib.request.urlopen(url, timeout=20) as r:
            data = r.read()
        verdict = 'OK con data' if len(data) > 1000 else f'TRANSPARENTE ({len(data)}B) ❌'
        print(f'{y:<6}{len(data):>14}{verdict:>30}')
    except urllib.error.HTTPError as e:
        print(f'{y:<6}   HTTP {e.code}: {e.reason}')
    except Exception as e:
        print(f'{y:<6}   ERROR: {type(e).__name__}: {str(e)[:50]}')

Año       Tile bytes                     Veredicto
2020             334         TRANSPARENTE (334B) ❌
2022             334         TRANSPARENTE (334B) ❌
2024             334         TRANSPARENTE (334B) ❌
